# Optimización de Hiperparámetros


## Objetivo

Aplicar GridSearchCV y RandomizedSearchCV para encontrar la mejor configuración de hiperparámetros en un modelo de regresión de tiempo de entrega.

## 1. Preparación del entorno

En esta sección cargamos las librerías y módulos necesarios para realizar la optimización de hiperparámetros.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import importlib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

from src import hyperparameter_tuning as ht

importlib.reload(ht)

ModuleNotFoundError: No module named 'src'

### 1.1 Importar librerías y módulos del proyecto

Cargamos las dependencias necesarias para preprocesar los datos, configurar el modelo y ejecutar la búsqueda de hiperparámetros.

In [ ]:
DATA_PATH = "../data/processed/logistica_clean.csv"
TARGET = "target_tiempo_entrega"
RANDOM_STATE = 42
TEST_SIZE = 0.2

df = pd.read_csv(DATA_PATH)
df.head()

## 2. Carga y revisión de datos

Cargamos el dataset procesado y verificamos su estructura básica.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

numeric_features, categorical_features

## 3. Selección de variables

Preparamos las variables predictoras y la variable objetivo, y separamos las columnas numéricas y categóricas.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

## 4. Definición del pipeline

Construimos el preprocesador y la tubería que aplicará escalado y codificación antes del modelo.

In [ ]:
model = RandomForestRegressor(random_state=RANDOM_STATE)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

## 5. División de entrenamiento y prueba

Separamos el conjunto en entrenamiento y prueba para evaluar la capacidad de generalización del modelo.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

In [ ]:
param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 5, 10, 15],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4]
}

## 6. Búsqueda de hiperparámetros

Definimos el espacio de búsqueda de parámetros para el modelo RandomForestRegressor y ejecutamos GridSearchCV.

In [ ]:
grid_search = ht.run_grid_search(
    model=pipeline,
    param_grid=param_grid,
    X_train=X_train,
    y_train=y_train,
    scoring="neg_mean_absolute_error",
    cv=5
)

## 7. Resultados de la optimización

Revisamos los mejores hiperparámetros encontrados y evaluamos el rendimiento del modelo optimizado con el conjunto de prueba.

In [ ]:
grid_search.best_params_

In [ ]:
grid_search.best_score_

In [ ]:
best_model = grid_search.best_estimator_

regression_metrics = ht.evaluate_regression_model(
    model=best_model,
    X_test=X_test,
    y_test=y_test
)

regression_metrics

## 8. Guardar resultados

Almacenamos el modelo final y los resultados de la búsqueda para su uso posterior y comparación con otras configuraciones.

In [ ]:
ht.save_best_model(
    model=best_model,
    output_path="../models/trained_models/best_supervised_model.pkl"
)

In [ ]:
ht.save_tuning_results(
    search_object=grid_search,
    output_path="../results/metrics/gridsearch_random_forest_results.csv"
)

In [ ]:
ht.summarize_best_params(
    search_object=grid_search,
    model_name="RandomForestRegressor",
    output_path="../results/metrics/best_hyperparameters.csv"
)

## Análisis del impacto

Después de aplicar GridSearchCV, comparamos el rendimiento del modelo optimizado con el modelo base. Una reducción en el MAE indica que el modelo ajustado comete un menor error promedio en la predicción del tiempo de entrega.

La optimización debe buscar un equilibrio entre los siguientes riesgos:

- Overfitting: cuando el modelo se ajusta demasiado a los datos de entrenamiento y pierde capacidad de generalizar.
- Underfitting: cuando el modelo es demasiado simple y no captura los patrones del problema.